In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

### Loading and Cleaning the Dataset

In [2]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [3]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Converting Text to Word-Count Features:

Naive Bayes (like almost every ML model) can't operate on raw text — it needs numeric features. `CountVectorizer` converts each message into a vector of word counts: one column per unique word in the vocabulary, with each cell holding how many times that word appears in that message. This is exactly the kind of feature Multinomial NB is built for.

In [4]:
X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Training matrix shape:", X_train_counts.shape)

Vocabulary size: 7741
Training matrix shape: (4457, 7741)


### Fitting Multinomial Naive Bayes:

In [5]:
mnb = MultinomialNB()
mnb.fit(X_train_counts, y_train)

y_pred = mnb.predict(X_test_counts)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label='spam'))
print("Recall:", recall_score(y_test, y_pred, pos_label='spam'))
print("F1-Score:", f1_score(y_test, y_pred, pos_label='spam'))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9829596412556054
Precision: 0.9642857142857143
Recall: 0.9060402684563759
F1-Score: 0.9342560553633218

Confusion Matrix:
 [[961   5]
 [ 14 135]]

Classification Report:
               precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



### Why Log-Probabilities Matter Here:

With a vocabulary of thousands of words, classifying even one message means multiplying together **hundreds** of small probabilities (one per word in that message). Each individual $P(x_i \mid y)$ is a small fraction — multiplying enough of them together causes **numerical underflow**: the result becomes so close to zero that floating-point precision can't represent it, and it silently rounds to exactly 0.

The fix is to work in **log-space** instead, since logarithms turn multiplication into addition:

$$\log P(y \mid X) \propto \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y)$$

Comparing sums instead of products gives the identical classification result, since $\log$ is a monotonic function — whichever class has the highest raw product also has the highest log-sum. sklearn's `MultinomialNB` does this internally by default; the cell below inspects the actual log-probabilities it computed, confirming they're stored and compared in log-space rather than as raw probabilities.

In [6]:
# feature_log_prob_ stores log(P(word | class)) directly — this IS the log-space representation in action
print("Shape of learned log-probabilities:", mnb.feature_log_prob_.shape)
print("(2 classes x vocabulary size)")

# Example: log-probability of the first 5 vocabulary words, per class
print("\nFirst 5 words' log-probabilities per class:\n", mnb.feature_log_prob_[:, :5])

# predict_log_proba gives log(P(y|X)) per message directly, avoiding underflow
log_probs = mnb.predict_log_proba(X_test_counts[:5])
print("\nLog-probabilities for first 5 test messages (ham, spam):\n", log_probs)

Shape of learned log-probabilities: (2, 7741)
(2 classes x vocabulary size)

First 5 words' log-probabilities per class:
 [[-10.97090153 -10.97090153 -10.27775435 -10.97090153 -10.97090153]
 [ -7.90728361  -6.80867132  -9.98672515  -9.29357797  -9.29357797]]

Log-probabilities for first 5 test messages (ham, spam):
 [[-1.05652356e-06 -1.37605272e+01]
 [-2.88808139e-05 -1.04523475e+01]
 [-1.82217139e-02 -4.01423835e+00]
 [-9.36625989e-06 -1.15784014e+01]
 [-6.95586409e-05 -9.57337519e+00]]


### Manual Demonstration of Underflow (Without Log-Space):

To make the underflow problem concrete: multiplying just a modest number of small probabilities together already collapses to zero in raw floating-point space, while the log-space equivalent stays perfectly representable.

In [ ]:
small_probs = np.random.uniform(0.001, 0.01, 500)

raw_product = np.prod(small_probs)
log_sum = np.sum(np.log(small_probs))

print("Raw product (underflows to 0):", raw_product)
print("Sum of logs (stays representable):", log_sum)
print("Exponentiating the log-sum back:", np.exp(log_sum))

Raw product (underflows to 0): 0.0
Sum of logs (stays representable): -2667.264922501711
Exponentiating the log-sum back: 0.0


### Handling Class Imbalance — A Preview of Complement NB:

This dataset is imbalanced (4,825 ham vs. 747 spam — about 6.5:1). Checking precision/recall separately (rather than only accuracy) matters here, since a model that just predicts "ham" for everything would already score ~87% accuracy while being useless. The next notebook (Complement Naive Bayes) is specifically designed to handle this kind of imbalance more robustly than plain Multinomial NB.

In [8]:
baseline_accuracy = (y_test == 'ham').mean()
print(f"'Always predict ham' baseline accuracy: {baseline_accuracy:.4f}")
print(f"Multinomial NB actual accuracy:         {accuracy_score(y_test, y_pred):.4f}")

'Always predict ham' baseline accuracy: 0.8664
Multinomial NB actual accuracy:         0.9830


### Summary:

- Multinomial NB models word counts per class using a multinomial distribution, with Laplace Smoothing applied by default to avoid zero-probability words.
- Working in log-space (as sklearn does internally) avoids numerical underflow when combining many small probabilities — essential for any text classification problem with a non-trivial vocabulary.
- Accuracy alone is misleading on this imbalanced dataset — precision/recall/F1 on the minority (spam) class give a much clearer picture of real performance.